## Setup
Config for where the raw CSVs live and where Bronze output goes, plus the imports used across this notebook. `CATALOG`/`SCHEMA` point at the Unity Catalog Volume for this project.

In [0]:
from pyspark.sql.types import StructField, StringType, StructType
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "transfermarkt"

## Ingest the 11 straightforward tables
These 11 read cleanly under Spark's automatic type inference, no known issues. Each is read as CSV, written out as Parquet, and its row count is printed as a sanity check against the dataset's expected scale.

In [0]:
tables = [
    "competitions", "clubs", "players", "appearances",
    "player_valuations", "club_games", "game_events",
    "game_lineups", "transfers", "countries", "national_teams",
]

for t in tables:
    raw_path = f"/Volumes/{CATALOG}/{SCHEMA}/raw/{t}.csv"
    bronze_path = f"/Volumes/{CATALOG}/{SCHEMA}/bronze/{t}"

    df = spark.read.csv(raw_path, header=True, inferSchema=True)
    df.write.mode("overwrite").parquet(bronze_path)
    print(f"{t}: {df.count()} rows")

## `games`, handled separately
Spark's schema inference misreads `aggregate` (the two-legged tie scoreline, e.g. "3:1") as a time-of-day value — the colon looks like a time separator to the inferencer. Left uncorrected, this silently turns real scorelines into fake timestamps instead of failing loudly, which is worse than a null: it looks valid. Fix: infer the schema once, override just that column to `StringType`, then re-read with the corrected schema.

In [0]:
raw_path = f"/Volumes/{CATALOG}/{SCHEMA}/raw/games.csv"
bronze_path = f"/Volumes/{CATALOG}/{SCHEMA}/bronze/games"

# infer the schema once, then override just the column Spark misreads as a time value
inferred = spark.read.csv(raw_path, header=True, inferSchema=True).schema
fixed_schema = StructType([
    StructField(f.name, StringType(), f.nullable) if f.name == "aggregate" else f
    for f in inferred.fields
])

games = spark.read.csv(raw_path, header=True, schema=fixed_schema)
games.write.mode("overwrite").parquet(bronze_path)
print(f"games: {games.count()} rows")

## Verify the fix
Confirms `aggregate` now holds real scorelines (e.g. `0:1`, `3:1`) instead of the fake `00:01:00`-style timestamps from the original bug.

In [0]:
def bronze(t):
    return spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/bronze/{t}")

bronze("games").select("aggregate").filter(F.col("aggregate").isNotNull()).show(5)